![](https://cdn-images-1.medium.com/max/800/1*8R1V4E00DJW1Vr9bh1hqkg.png)


[Python datatable](https://datatable.readthedocs.io/en/latest/?badge=latest) is a library that implements a wide (and growing) range of operators for manipulating two-dimensional data frames. It focuses on: **big data support, high performance, both in-memory and out-of-memory datasets, and multithreaded algorithms**. Datatable's powerful API is similar to R data.table's, and it strives in providing friendlier and intuitive API experience with helpful error messages to accelerate problem-solving.

* Some of the notable features of datatable are:
* Efficient multi-threaded algorithms
* Memory-thrifty
* Memory-mapped on-disk datasets
* Native C++ implementation
* Fully Opensourced

In this notebook, we shall try to understand about data wrangling with datatable via a banking loan scenario using a subset of the Fannie Mae dataset. This notebook shows how to munge loan-level data and  obtain basic insights.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Any results you write to the current directory are saved as output.

In [ ]:
# Install the datatable package : 

!pip install pip --upgrade
!pip install datatable # Turn 'ON'the internet option to install the latest version

In [ ]:
#Import the datatable package
import datatable as dt
print(dt.__version__)

## Dataset

![](https://images.unsplash.com/photo-1560518883-ce09059eeffa?ixlib=rb-1.2.1&ixid=eyJhcHBfaWQiOjEyMDd9&auto=format&fit=crop&w=966&q=80)

Dataset is derived from [Fannie Mae's Single-Family Loan Performance Data(SFLP)](http://www.fanniemae.com/portal/funding-the-market/data/loan-performance-data.html) with all rights reserved by Fannie Mae. For the full raw dataset, you will need to register on the Fannie Mae's site. As of this writing, the most recent data set that's available is from the third quarter of 2019. However, this article uses the dataset for the third quarter of 2014 which can be downloaded from [here](https://app-freddie-mac.s3.amazonaws.com/fannie-mae/2014Q3.zip).

The downloaded dataset comprises of two files called **Acquisition.txt** and **Performance.txt**:

* **The acquisition data**: contains personal information for each of the borrowers, including an individual's debt-to-income ratio, credit score, and loan amount, among several other things.

* **The performance data**: contains information regarding loan payment history, and whether or not a borrower ended up defaulting on their loan.

Additional information regarding the contents of these two files can also be found on the website in the form of [Glossary](https://s3.amazonaws.com/dq-blog-files/lppub_glossary.pdf)  and [Columns](https://s3.amazonaws.com/dq-blog-files/lppub_file_layout.pdf) containing the columns in the Acquisition and Performance files

There are twwo reasons for using this dataset:
1. The data size is ideal to demonstrate the capabilities of the datatable library.
2. The dataset requires some preprocessng which can be then demonstrated via the datatable library.


## Objective
Our goal would be to predict from this data, with some accuracy, those borrowers who are most at risk of defaulting on their mortgage loans.

## Loading data

The frame doesn't have the column headers which we will need to enter manually from the [columns](https://s3.amazonaws.com/dq-blog-files/lppub_file_layout.pdf) file.

In [ ]:
col_acq = ['LoanID','Channel','SellerName','OrInterestRate','OrUnpaidPrinc','OrLoanTerm','OrDate','FirstPayment','OrLTV','OrCLTV','NumBorrow','DTIRat','CreditScore','FTHomeBuyer','LoanPurpose','PropertyType','NumUnits','OccStatus','PropertyState','Zip','MortInsPerc','ProductType','CoCreditScore','MortInsType','RelocationMort']

col_per = ['LoanID','MonthRep','Servicer','CurrInterestRate','CAUPB','LoanAge','MonthsToMaturity','AdMonthsToMaturity','MaturityDate','MSA','CLDS','ModFlag','ZeroBalCode','ZeroBalDate','LastInstallDate','ForeclosureDate','DispositionDate','ForeclosureCosts','PPRC','AssetRecCost','MHRC','ATFHP','NetSaleProceeds','CreditEnhProceeds','RPMWP','OFP','NIBUPB','PFUPB','RMWPF',  'FPWA','SERVICING ACTIVITY INDICATOR']

In [ ]:
# Reading the data into a Frame object

df_acq = dt.fread('../input/Acquisition_2014Q3.txt',columns=col_acq)
df_per = dt.fread('../input/Performance_2014Q3.txt', columns=col_per)

The `fread()` function above is both powerful and extremely fast.It is 5x times faster than pandas.read_csv() It can automatically detect and parse parameters from the majority of text files, load data from .zip archives or URLs, read Excel files, and much more.
Let's check the shape of both the dataframes

In [ ]:
print(df_acq.shape)
print(df_per.shape)


Viewing the First few rows of the acquisitions and Performance Dataframe. Unlike Pandas, the .head() function displays the first 10 rows of a frame although you can specify the no. of rows to be displayed in the parenthesis

In [ ]:
df_acq.head() 

In [ ]:
df_per.head(5)

![](https://image.slidesharecdn.com/datatable-h2oworld-sf-190211190456/95/machine-learning-and-data-munging-in-h2o-driverless-ai-with-datatable-7-638.jpg?cb=1559075937)

The colour signifies the datatype where red denotes string, green denotes int and blue stands for float.

## Data Preprocessing
Data Tables like dataframes are columnar data structures. In datatable, the primary vehicle for all these operations is the square-bracket notation inspired by traditional matrix indexing but with more functionalities.

![](https://miro.medium.com/max/723/1*PI7NS0fRaqmY2rXnrmL_Lg.png)


In the performance data, we are really only interested in the LoanID and ForeclosureDate columns, as this will give us the borrower identification number and whether or not they ended up defaulting. 

In [ ]:
# Selecting only the LoanID and the ForeclosureDate column and discarding the rest
df_per = df_per[:,['LoanID','ForeclosureDate']]
df_per.head()

### Removing Duplicate LoanIDs from the Performance Dataframe
The Loan ID contains duplicated entities. Let's get rid of them.

In [ ]:
# Displaying only the unique Loan IDs in the Performance dataset
dt.unique(df_per[:,"LoanID"]).head(5)

### Grouping
Filtering the frame so that it has only unique IDs, is equivalent to grouping. In that case we can achieve the same by:

In [ ]:
# Filtering
df_per = df_per[-1:,:, dt.by(dt.f.LoanID)]
df_per.head(5)

The **[f-expression](https://datatable.readthedocs.io/en/latest/f-expressions.html)** supports arithmetic operations as well as various mathematical and aggregate functions.

### Joining the Acquisition and performance datasets
We will perform an inner join on the Acquisition and Performance dataframes using the LoanID column. The resulting dataframe, df, will contain the ForeclosureDate column, and will be our target variable. For clarity, we will also rename this column as Will_Default.

In [ ]:
df_per.names = ['LoanID','Will_Default']
df_per.key = 'LoanID'
df= df_acq[:,:,dt.join(df_per)]

In [ ]:
# logical types
df[:,'Will_Default'].ltypes

In [ ]:
# Grouping by the 'Will Deafult' column
df[1:,:, dt.by(dt.f.Will_Default)].head(5)

### Formatting the Target Column
In the Will_Default column, a '1'' is placed next to any borrower that was found to have defaulted, and a '0' is placed next to any borrower that has not defaulted i.e who has paid the loan on some date.

In [ ]:
# Replacing the dates in the Will_Default column with '0' and null values with 1
df[:,'Will_Default'] = df[:, {'Will_Default': dt.f['Will_Default']==""}]
df.head(5)

In [ ]:
df.shape

The dataframe has 394356 rows and 26 columns, and contains information regarding loan interest rate, payment dates, property state, and the last few digits of each property ZIP code, among several other things.From here the dataframe is ready to be fed into a model for training purposes. Once can also convert it into a Pandas dataframe, csv file or into a binary.jay file.

In [ ]:
#df.to_pandas()
#df.to_csv("out.csv")
#df.to_jay("data.jay")

## References and useful resources
Here are some of the resources that will be useful to understand and learn more about datatable’s features:
* [DatatableTon](https://github.com/vopani/datatableton) — 100 datatable exercises over different sections structured as a course or tutorials to teach and learn for beginners, intermediates as well as experts
* [An Overview of Python’s Datatable package](https://towardsdatascience.com/an-overview-of-pythons-datatable-package-5d3a97394ee9)
* [Documentation](https://datatable.readthedocs.io/en/latest/quick-start.html)
* [EDA with Python datatable](https://lnkd.in/fdVqJVr)
* [H2O Light Fast Implementation of FTRL, 1 epoch 7sec](https://lnkd.in/fJqcTYy)
* [Getting started with Python datatable](https://lnkd.in/fPaezRp)